In [ ]:
# Needed imports
import time
import warnings
import os
from __future__ import annotations
import pyzx as zx

import argparse
from dataclasses import dataclass
from pathlib import Path
from time import perf_counter
from typing import Any

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector

#os.environ["BLUEQUBIT_PPS_DO_NO_USE_PARALLEL_COMPUTE"] = "1
os.environ["BLUEQUBIT_DEQART_INTERNAL_DISABLE_STRICT_VALIDATIONS"] = "1"
warnings.filterwarnings("ignore", category=UserWarning)


import bluequbit
bq = bluequbit.init()
# Call with API key

In [121]:
# Load qasm
qc = QuantumCircuit.from_qasm_file("P8_bold_peak.qasm")

print(qc.count_ops())

OrderedDict({'u3': 2696, 'cz': 1319})


In [ ]:
# Basic MPS for Problem 3-4
results_mps = bq.run(
    qc,
    device="mps.cpu",
    shots=10000,
    options={"mps_bond_dimension": 45},
)

print(f"MPS simulation runtime: {results_mps.run_time_ms} ms")

In [ ]:
# Find most frequent and majority-vote bitstrings
counts = results_mps.get_counts()
top10_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:10]

total_shots = sum(counts.values())
n_bits = len(next(iter(counts)))

bit_sums = [0] * n_bits
for bitstring, count in counts.items():
    for i, bit in enumerate(bitstring):
        bit_sums[i] += int(bit) * count

majority_bitstring = ''.join('1' if s > total_shots / 2 else '0' for s in bit_sums)

print("Top 10 most frequent bitstrings from MPS Simulation:\n")
for bitstring, count in top10_counts:
    print(f"  {bitstring}: {count}")

print(f"\nMost frequent bitstring:  {top10_counts[0][0]}")
print(f"Peak bitstring with majority vote): {majority_bitstring}")

In [113]:
# Marginal Attack
qc.remove_final_measurements(inplace=True)

# Print basic circuit information
print(f"Number of qubits:        {qc.num_qubits}")
print(f"Gate counts:             {qc.count_ops()}")
print(f"Total instructions:      {len(qc.data)}")

Number of qubits:        60
Gate counts:             OrderedDict({'cz': 1082, 'u': 1006})
Total instructions:      2088


In [114]:
# Based off of Bluequbit tutorial
def make_single_qubit_Z_ops(qc):
    """
    Construct the list of single-qubit Z observables for the circuit.
    Each entry is a list containing a single (Pauli_string, coefficient) tuple.
    """
    n = qc.num_qubits
    Z_ops = []
    for i in range(n):
        pauli_str = ''.join('Z' if j == i else 'I' for j in range(n))
        # Reverse due to Qiskit's little-endian ordering of qubits
        pauli_str = pauli_str[::-1]
        Z_ops.append([(pauli_str, 1.0)])
    return Z_ops

# Generate all single-qubit Z observables for this circuit
Z_ops = make_single_qubit_Z_ops(qc)
assert len(Z_ops) == qc.num_qubits

In [ ]:
# Use MPS
z_expvals_mps = bq.run(
    qc,
    device="mps.cpu",
    pauli_sum=Z_ops,
    options={"mps_bond_dimension": 53},
).expectation_value

z_expvals_mps = np.array(z_expvals_mps)
# Reconstruct the bitstring:
bits = (z_expvals_mps < 0).astype(int)  # 1 if negative, else 0
peak_bitstring_mps = ''.join(str(b) for b in bits)
print(peak_bitstring_mps)

In [ ]:
# Approximate Transpilation
z_expvals = []

qc_tr = transpile(
    qc,
    basis_gates=set(qc.count_ops()),
    approximation_degree=0.995,
    optimization_level=3,
)

In [ ]:
# PPS Marginal-Attack

# Print the gate counts of the transpiled circuit to see how much simplification occurred
print("Gate counts after approximate transpilation:")
print(qc_tr.count_ops())

pps_options = {
    "pauli_path_truncation_threshold": 1e-2,
    "pauli_path_circuit_transpilation_level": 1,
}

# Compute ⟨Z_i⟩ for each qubit
for i, Z_op in enumerate(Z_ops):
    result = bq.run(qc_tr, device="pauli-path", pauli_sum=Z_op, options=pps_options)
    exp_val = result.expectation_value
    z_expvals.append(exp_val)
    print(f"qubit {i:2d}:  <Z_{i}> = {exp_val:.6f}")

In [ ]:
# Problem 7 Graph Analysis

@dataclass
class ComponentResult:
    qubits: list[int]
    gate_counts: dict[str, int]
    depth: int
    peak_probability: float
    peak_bitstring_qorder: str
    peak_bitstring_qiskit_order: str


@dataclass
class Problem7Result:
    num_qubits: int
    component_results: list[ComponentResult]
    candidate_bitstring_qorder: str
    candidate_bitstring_qiskit_order: str
    total_runtime_s: float


def load_circuit(qasm_path: str | Path) -> QuantumCircuit:
    return QuantumCircuit.from_qasm_file(str(qasm_path))


def remove_measurements(qc: QuantumCircuit) -> QuantumCircuit:
    work = qc.copy()
    work.remove_final_measurements(inplace=True)
    return work


def cz_connected_components(qc: QuantumCircuit) -> list[list[int]]:
    adjacency: dict[int, set[int]] = {i: set() for i in range(qc.num_qubits)}
    for inst in qc.data:
        if inst.operation.name != "cz":
            continue
        a, b = sorted(q._index for q in inst.qubits)
        adjacency[a].add(b)
        adjacency[b].add(a)

    seen: set[int] = set()
    components: list[list[int]] = []
    for qubit in range(qc.num_qubits):
        if qubit in seen:
            continue
        stack = [qubit]
        seen.add(qubit)
        component: list[int] = []
        while stack:
            node = stack.pop()
            component.append(node)
            for neighbor in adjacency[node]:
                if neighbor not in seen:
                    seen.add(neighbor)
                    stack.append(neighbor)
        components.append(sorted(component))
    return components


def extract_component_circuit(qc: QuantumCircuit, component_qubits: list[int]) -> QuantumCircuit:
    mapping = {qubit: i for i, qubit in enumerate(component_qubits)}
    subcircuit = QuantumCircuit(len(component_qubits))
    for inst in qc.data:
        qubits = [q._index for q in inst.qubits]
        if not qubits:
            continue
        if all(qubit in mapping for qubit in qubits):
            subcircuit.append(inst.operation, [mapping[qubit] for qubit in qubits], [])
    return subcircuit


def solve_component_with_statevector(subcircuit: QuantumCircuit) -> tuple[str, float]:
    statevector = Statevector.from_instruction(subcircuit)
    probabilities = np.abs(statevector.data) ** 2
    peak_index = int(np.argmax(probabilities))
    peak_probability = float(probabilities[peak_index])
    peak_bitstring_qiskit_order = format(peak_index, f"0{subcircuit.num_qubits}b")
    peak_bitstring_qorder = peak_bitstring_qiskit_order[::-1]
    return peak_bitstring_qorder, peak_probability


def solve_component_with_bluequbit(bq: Any, subcircuit: QuantumCircuit) -> tuple[str, float]:
    result = bq.run(subcircuit)
    counts = result.get_counts()
    peak_bitstring_qiskit_order, peak_probability = max(
        counts.items(), key=lambda item: item[1]
    )
    return peak_bitstring_qiskit_order[::-1], float(peak_probability)


def solve_component_peak(
    subcircuit: QuantumCircuit,
    *,
    bq: Any | None = None,
) -> tuple[str, float]:
    if bq is None:
        return solve_component_with_statevector(subcircuit)
    return solve_component_with_bluequbit(bq, subcircuit)


def attack_problem7(
    *,
    qasm_path: str | Path = "P7_rolling_ridge.qasm",
    bq: Any | None = None,
) -> Problem7Result:
    start = perf_counter()
    qc = remove_measurements(load_circuit(qasm_path))
    components = cz_connected_components(qc)

    global_bits_by_qubit = ["0"] * qc.num_qubits
    component_results: list[ComponentResult] = []

    for component_qubits in components:
        subcircuit = extract_component_circuit(qc, component_qubits)
        peak_bitstring_qorder, peak_probability = solve_component_peak(subcircuit, bq=bq)
        for qubit, bit in zip(component_qubits, peak_bitstring_qorder):
            global_bits_by_qubit[qubit] = bit

        component_results.append(
            ComponentResult(
                qubits=component_qubits,
                gate_counts={k: int(v) for k, v in subcircuit.count_ops().items()},
                depth=subcircuit.depth(),
                peak_probability=peak_probability,
                peak_bitstring_qorder=peak_bitstring_qorder,
                peak_bitstring_qiskit_order=peak_bitstring_qorder[::-1],
            )
        )

    return Problem7Result(
        num_qubits=qc.num_qubits,
        component_results=component_results,
        candidate_bitstring_qorder="".join(global_bits_by_qubit),
        candidate_bitstring_qiskit_order="".join(reversed(global_bits_by_qubit)),
        total_runtime_s=perf_counter() - start,
    )

result = attack_problem7(
    qasm_path="P7_rolling_ridge.qasm",
    bq=None,
)

print("Candidate bitstring (q-order):    ", result.candidate_bitstring_qorder)
print("Candidate bitstring (Qiskit order):", result.candidate_bitstring_qiskit_order)
print(f"Runtime: {result.total_runtime_s:.2f}s")
print()
for comp in result.component_results:
    print(
        f"  qubits={comp.qubits}",
        f"peak_prob={comp.peak_probability:.4f}",
        f"peak_qorder={comp.peak_bitstring_qorder}",
    )

In [ ]:
# Used given Github repo for problem 8-9 using better pyZX optimization

def load_qasm(path: str) -> str:
    """Load QASM file and remove barrier lines."""
    with open(path, "r") as f:
        qasm_str = f.read()
    # Remove barriers (PyZX doesn't like them)
    qasm_str = "\n".join(
        line for line in qasm_str.split("\n") if not line.strip().startswith("barrier")
    )
    return qasm_str


def pyzx_simplify(qasm_str: str, strategy: str = "clifford_simp") -> str:
    """
    Apply PyZX simplification.

    Strategies:
    - 'clifford_simp': usually robust, often simplifies well
    - 'full_reduce': aggressive ZX-calculus reduction
    - 'spider_simp': similar family to clifford_simp
    - 'teleport_reduce': teleportation-based reduction
    """
    circuit = zx.Circuit.from_qasm(qasm_str)
    print(f"  Original: {circuit.qubits} qubits, {len(circuit.gates)} gates")

    g = circuit.to_graph()

    if strategy == "clifford_simp":
        zx.simplify.clifford_simp(g)
    elif strategy == "spider_simp":
        zx.simplify.spider_simp(g)
    elif strategy == "full_reduce":
        zx.full_reduce(g)
    elif strategy == "teleport_reduce":
        zx.teleport_reduce(g)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    # IMPORTANT: do NOT force up_to_perm=False
    circ_reduced = zx.extract_circuit(g)
    print(f"  After {strategy}: {len(circ_reduced.gates)} gates")

    return circ_reduced.to_qasm()


def qiskit_optimize(
    qasm_str: str,
    optimization_level: int = 3,
    approximation_degree: float | None = None,
) -> QuantumCircuit:
    """Apply Qiskit transpiler optimization."""
    qc = QuantumCircuit.from_qasm_str(qasm_str)
    print(f"  Before transpile: depth={qc.depth()}, size={qc.size()}")

    # CRITICAL: measure before transpile
    qc.measure_all()

    optimized = transpile(
        qc,
        basis_gates=["u3", "cx"],
        optimization_level=optimization_level,
        approximation_degree=approximation_degree,
    )

    approx_str = f", approx={approximation_degree}" if approximation_degree else ""
    print(
        f"  After transpile (level {optimization_level}{approx_str}): "
        f"depth={optimized.depth()}, size={optimized.size()}"
    )
    return optimized


def run_mps(qc: QuantumCircuit, shots: int = 2000, bond_dim: int = 128) -> dict[str, int]:
    """Run simulation using Qiskit Aer MPS backend."""
    backend = AerSimulator(method="matrix_product_state")
    backend.set_options(matrix_product_state_max_bond_dimension=bond_dim)

    print(f"  MPS backend: CPU, bond_dim={bond_dim}")

    t0 = time.perf_counter()
    job = backend.run(qc, shots=shots)
    counts = job.result().get_counts()
    dt = time.perf_counter() - t0
    print(f"  Sample time: {dt:.2f}s")

    return counts


def reconstruct_bitstring(measurement: str, n_qubits: int) -> str:
    """
    Normalize Aer count keys to a clean bitstring.

    - Removes spaces (Aer may insert spaces for readability on large bitstrings)
    - Truncates to n_qubits bits (defensive; should already match)
    """
    measurement = measurement.replace(" ", "")
    if len(measurement) > n_qubits:
        measurement = measurement[:n_qubits]
    return measurement


def solve(
    qasm_path: str,
    shots: int = 2000,
    bond_dim: int = 128,
    strategy: str = "clifford_simp",
    opt_level: int = 3,
    approx_degree: float | None = None,
    skip_pyzx: bool = False,
) -> dict:
    """Solve a circuit to find its peak bitstring."""
    print(f"\n{'='*60}")
    print(f"Solving: {qasm_path}")
    approx_str = f", approx={approx_degree}" if approx_degree else ""
    pyzx_str = " (skip PyZX)" if skip_pyzx else f", strategy={strategy}"
    print(f"bond_dim={bond_dim}, shots={shots}{approx_str}{pyzx_str}")
    print(f"{'='*60}")

    t_start = time.perf_counter()

    print("\n[1] Loading QASM...")
    _ = load_qasm(qasm_path)  # keeps barrier stripping behavior explicit

    # Re-dump through Qiskit for a clean canonical QASM string
    qc_orig = QuantumCircuit.from_qasm_file(qasm_path)
    qasm_str = qasm2.dumps(qc_orig)
    n_qubits = qc_orig.num_qubits
    print(
        f"  Loaded: {qc_orig.num_qubits} qubits, depth={qc_orig.depth()}, size={qc_orig.size()}"
    )

    if skip_pyzx:
        print("\n[2] Skipping PyZX (using Qiskit transpile only)...")
        qasm_opt = qasm_str
    else:
        print(f"\n[2] PyZX Simplification ({strategy})...")
        try:
            qasm_opt = pyzx_simplify(qasm_str, strategy=strategy)
        except Exception as e:
            print(f"  PyZX failed: {e}")
            print("  Falling back to original circuit...")
            qasm_opt = qasm_str

    print(f"\n[3] Qiskit Transpile (level {opt_level})...")
    qc = qiskit_optimize(
        qasm_opt, optimization_level=opt_level, approximation_degree=approx_degree
    )

    print("\n[4] MPS Simulation...")
    counts = run_mps(qc, shots=shots, bond_dim=bond_dim)

    peak_raw = max(counts, key=counts.get)
    peak_count = counts[peak_raw]
    peak_bitstring = reconstruct_bitstring(peak_raw, n_qubits)

    dt_total = time.perf_counter() - t_start

    print(f"\n{'='*60}")
    print("RESULTS")
    print(f"{'='*60}")
    print(f"Total time: {dt_total:.2f}s")
    print(f"\nPeak bitstring: {peak_bitstring}")
    print(f"Count: {peak_count}/{shots} ({100*peak_count/shots:.1f}%)")

    print("\nTop 5:")
    for bs, c in sorted(counts.items(), key=lambda x: -x[1])[:5]:
        reconstructed = reconstruct_bitstring(bs, n_qubits)
        print(f"  {reconstructed}: count={c}")

    return {
        "bitstring": peak_bitstring,
        "count": peak_count,
        "total_shots": shots,
        "time": dt_total,
        "strategy": strategy,
        "bond_dim": bond_dim,
        "skip_pyzx": skip_pyzx,
        "approx_degree": approx_degree,
    }


result = solve("P9_grand_summit.qasm", shots=10000, bond_dim=256, strategy="full_reduce")